# 01 — Binance Market Data Exploration

## Goal

Understand the structure of Binance market data before building the data pipeline.

Tasks:

- Inspect Binance API structure
- Understand trade messages
- Understand order book depth updates
- Examine timestamps and ordering
- Identify fields needed for limit order book reconstruction

This notebook is **exploratory**.  
Production code will later move into:



---

# 1. Exchange and Market Selection

For this project we use:

Exchange: Binance
Pair: BTCUSDT

- High liquidity
- Access to limit order book
- Good api support
- Free to use




# Load Required Libraries

In [65]:
import pandas as pd
import numpy as np
from binance.client import Client
import time
import black

pd.set_option("display.max_columns", None)

# Get Order Book
I want to just get data one time to allow for reproducible results


In [66]:
client = Client()

Check latency, since binance doesnt give time with the non websocket depth snapshots

In [67]:
t0 = int(time.time() * 1000)

depth = client.get_order_book(symbol="BTCUSDT")

t1 = int(time.time() * 1000)

rtt = t1 - t0
latency = rtt / 2
latency

118.5

Latency ~ 151 ms so using 1 second intervals for snapshots should in general be fine (missing intra second data) - future update is to add in websocket intrasecond info

In [68]:
depth_samples = []

for i in range(5):
    timestamp = int(time.time() * 1000)
    depth = client.get_order_book(symbol="BTCUSDT", limit=100)
    bids = depth["bids"]
    asks = depth["asks"]
    depth_samples.append(
        {
            "timestamp": timestamp,
            "lastUpdateId": depth["lastUpdateId"],
            "bids": depth["bids"],
            "asks": depth["asks"],
        }
    )
    time.sleep(1)
trades = client.get_recent_trades(symbol="BTCUSDT", limit=1000)

depth_samples



[{'timestamp': 1773103232775,
  'lastUpdateId': 89787578577,
  'bids': [['69154.00000000', '0.24116000'],
   ['69153.99000000', '0.00024000'],
   ['69153.91000000', '0.00008000'],
   ['69153.84000000', '0.00352000'],
   ['69153.75000000', '0.00008000'],
   ['69153.74000000', '0.00008000'],
   ['69153.69000000', '0.00024000'],
   ['69153.68000000', '0.05000000'],
   ['69153.28000000', '0.00016000'],
   ['69153.27000000', '0.08530000'],
   ['69152.52000000', '0.00016000'],
   ['69152.51000000', '0.13010000'],
   ['69152.40000000', '0.00008000'],
   ['69152.32000000', '0.00008000'],
   ['69152.16000000', '0.00008000'],
   ['69152.00000000', '0.00120000'],
   ['69151.54000000', '0.00030000'],
   ['69151.50000000', '0.00008000'],
   ['69150.94000000', '0.00008000'],
   ['69150.65000000', '0.00008000'],
   ['69150.19000000', '0.00008000'],
   ['69150.18000000', '0.00153000'],
   ['69150.16000000', '0.00008000'],
   ['69150.00000000', '0.00120000'],
   ['69149.97000000', '0.00008000'],
   ['6

Create bids df exploded in bids same with asks

In [69]:
bids_df = pd.DataFrame(depth_samples)
bids_df = bids_df[["timestamp", "lastUpdateId", "bids"]].explode("bids")

bids_df["bids_price"], bids_df["bids_volume"] = zip(*bids_df["bids"])

bids_df["bids_price"] = bids_df["bids_price"].astype(float)
bids_df["bids_volume"] = bids_df["bids_volume"].astype(float)

bids_df = bids_df.drop(columns="bids")
bids_df.head()

,timestamp,lastUpdateId,bids_price,bids_volume
0,1773103232775,89787578577,69154.000000,0.241160
0,1773103232775,89787578577,69153.990000,0.000240
0,1773103232775,89787578577,69153.910000,0.000080
0,1773103232775,89787578577,69153.840000,0.003520
0,1773103232775,89787578577,69153.750000,0.000080


In [70]:
asks_df = pd.DataFrame(depth_samples)
asks_df = asks_df[["timestamp", "lastUpdateId", "asks"]].explode("asks")
asks_df["asks_price"], asks_df["asks_volume"] = zip(*asks_df["asks"])

asks_df["asks_price"] = asks_df["asks_price"].astype(float)
asks_df["asks_volume"] = asks_df["asks_volume"].astype(float)

asks_df = asks_df.drop(columns="asks")
asks_df.head()

,timestamp,lastUpdateId,asks_price,asks_volume
0,1773103232775,89787578577,69154.010000,1.119930
0,1773103232775,89787578577,69154.020000,0.000160
0,1773103232775,89787578577,69154.430000,0.000160
0,1773103232775,89787578577,69154.440000,0.014260
0,1773103232775,89787578577,69155.020000,0.002000


now for both asks and bids gets the highest price for each timestamp

In [71]:
max_bid = bids_df.groupby("timestamp")["bids_price"].max().rename("max_bids")
max_ask = asks_df.groupby("timestamp")["asks_price"].max().rename("max_asks")
max_bid_vol = bids_df.groupby("timestamp")["bids_volume"].max().rename("max_bids_vol")
max_ask_vol = asks_df.groupby("timestamp")["asks_volume"].max().rename("max_asks_vol")

In [72]:
combined_df = pd.concat([max_bid, max_ask, max_bid_vol, max_ask_vol], axis=1)
combined_df.reset_index(inplace=True)
combined_df

,timestamp,max_bids,max_asks,max_bids_vol,max_asks_vol
0,1773103232775,69154.000000,69173.400000,2.385600,1.387990
1,1773103234010,69154.000000,69171.900000,2.385600,1.407990
2,1773103235244,69166.730000,69188.000000,3.773590,1.387990
3,1773103236482,69166.730000,69187.190000,3.773590,0.338090
4,1773103237717,69169.870000,69189.120000,3.871010,1.387990


Create features

- timestamp
- best_bid
- best_ask
- mid_price
- spread
- bid_volume
- ask_volume
- imbalance
- microprice
  - to do weighted microprice (maybe add in 5 layers to lob not just best price)
- trade_imbalance



In [73]:
combined_df["mid_price"] = (combined_df["max_bids"] + combined_df["max_asks"]) / 2
combined_df["spread"] = combined_df["max_asks"] - combined_df["max_bids"]

combined_df["imbalance"] = (
    combined_df["max_bids_vol"] - combined_df["max_asks_vol"]
) / (combined_df["max_bids_vol"] + combined_df["max_asks_vol"])


combined_df["liquidity"] = combined_df["max_bids_vol"] + combined_df["max_asks_vol"]
combined_df["microprice"] = (combined_df["max_bids_vol"]*combined_df["max_bids"] + combined_df["max_asks_vol"]*combined_df["max_asks"]) / combined_df["liquidity"]
combined_df["mid_price_change"] = combined_df["mid_price"].shift(-1) - combined_df["mid_price"]
combined_df

,timestamp,max_bids,max_asks,max_bids_vol,max_asks_vol,mid_price,spread,imbalance,liquidity,microprice,mid_price_change
0,1773103232775,69154.000000,69173.400000,2.385600,1.387990,69163.700000,19.400000,0.264366,3.773590,69161.135647,-0.750000
1,1773103234010,69154.000000,69171.900000,2.385600,1.407990,69162.950000,17.900000,0.257700,3.793590,69160.643581,14.415000
2,1773103235244,69166.730000,69188.000000,3.773590,1.387990,69177.365000,21.270000,0.462184,5.161580,69172.449673,-0.405000
3,1773103236482,69166.730000,69187.190000,3.773590,0.338090,69176.960000,20.460000,0.835547,4.111680,69168.412359,2.535000
4,1773103237717,69169.870000,69189.120000,3.871010,1.387990,69179.495000,19.250000,0.472147,5.259000,69174.950587,NaN


trade df

In [74]:
trades_df = pd.DataFrame(trades)
trades_df = trades_df.astype({
    "price": float,
    "qty": float,
    "quoteQty": float,
    "time": "int64",
    "isBuyerMaker": bool,
    "isBestMatch": bool,
})
trades_df

,id,price,qty,quoteQty,time,isBuyerMaker,isBestMatch
0,6085762643,69179.590000,0.000160,11.068734,1773103221414,True,True
1,6085762644,69179.590000,0.000080,5.534367,1773103221414,True,True
2,6085762645,69179.590000,0.000080,5.534367,1773103221414,True,True
3,6085762646,69179.590000,0.000080,5.534367,1773103221414,True,True
4,6085762647,69179.590000,0.000080,5.534367,1773103221414,True,True
...,...,...,...,...,...,...,...
995,6085763638,69178.990000,0.000080,5.534319,1773103239161,False,True
996,6085763639,69178.990000,0.000080,5.534319,1773103239161,False,True
997,6085763640,69178.990000,0.000080,5.534319,1773103239161,False,True
998,6085763641,69178.990000,0.000080,5.534319,1773103239161,False,True


I need to segment trades info into 5 time slots each for each second (need to add in buffer for latency)

In [75]:
pd.set_option("display.float_format", "{:.6f}".format)
trades_df_interval_masks = []

timestamps = combined_df["timestamp"].values
intervals = zip(timestamps[:-1],timestamps[1:])

for t0, t1 in intervals:
    mask = (trades_df["time"]>t0) & (trades_df["time"]<t1)
    trades_df_interval_masks.append(mask) 

    

Check to see if trades correct time intervals

In [76]:
trade_slice_1= trades_df[trades_df_interval_masks[0]]
        
trade_slice_1["time"].describe()

count               2.000000
mean    1773103233972.000000
std                22.627417
min     1773103233956.000000
25%     1773103233964.000000
50%     1773103233972.000000
75%     1773103233980.000000
max     1773103233988.000000
Name: time, dtype: float64

In [77]:
trade_slice_1

,id,price,qty,quoteQty,time,isBuyerMaker,isBestMatch
528,6085763171,69154.010000,0.083550,5777.817536,1773103233956,False,True
529,6085763172,69154.010000,0.002010,138.999560,1773103233988,False,True


In [78]:
for i, mask in enumerate(trades_df_interval_masks):
    print(i, mask.sum())

0 2
1 9
2 212
3 70


get trade features do for 1 slice then create function to do for all slices
- trade_count
- buy_count
- sell_count
- total_trade_volume
- buy_volume
- sell_volume
- trade_imbalance
- trade_flow_ratio
- avg_trade_size
- max_trade_size
- std_trade_size
- vwap


In [79]:
trade_count = len(trade_slice_1)
buy_count = len(trade_slice_1[ trade_slice_1["isBuyerMaker"]==True])
sell_count = len(trade_slice_1[ trade_slice_1["isBuyerMaker"]==False])

total_trade_volume = trade_slice_1["qty"].sum()

buy_volume = trade_slice_1.loc[trade_slice_1["isBuyerMaker"]==True,"qty"].sum()
sell_volume = trade_slice_1.loc[trade_slice_1["isBuyerMaker"]==False,"qty"].sum()

trade_imbalance = (buy_volume - sell_volume) / total_trade_volume

trade_flow_ratio = buy_volume/sell_volume

avg_trade_size = total_trade_volume / trade_count

max_trade_size = trade_slice_1["qty"].max()
min_trade_size = trade_slice_1["qty"].min()

std_trade_size = trade_slice_1["qty"].std()

vwap = (trade_slice_1["price"]*trade_slice_1["qty"]).sum() / total_trade_volume

trade_slice_1_features = {

    "trade_count": trade_count,
    "buy_count": buy_count,
    "sell_count": sell_count,
    "total_trade_volume": total_trade_volume,
    "buy_volume": buy_volume,
    "sell_volume": sell_volume,
    "trade_imbalance": trade_imbalance,
    "trade_flow_ratio": trade_flow_ratio,
    "avg_trade_size": avg_trade_size,
    "max_trade_size": max_trade_size,
    "min_trade_size": min_trade_size,
    "std_trade_size": std_trade_size,
    "vwap": vwap
}

trade_slice_1_features_df = pd.DataFrame([trade_slice_1_features])
trade_slice_1_features_df


,trade_count,buy_count,sell_count,total_trade_volume,buy_volume,sell_volume,trade_imbalance,trade_flow_ratio,avg_trade_size,max_trade_size,min_trade_size,std_trade_size,vwap
0,2,0,2,0.085560,0.000000,0.085560,-1.000000,0.000000,0.042780,0.083550,0.002010,0.057657,69154.010000


Combine tradefeatures with lob features

In [80]:
#
# 
trade_and_lob_features_slice_0_1 = pd.concat([combined_df.iloc[[0]],trade_slice_1_features_df],axis=1)

trade_and_lob_features_slice_0_1

,timestamp,max_bids,max_asks,max_bids_vol,max_asks_vol,mid_price,spread,imbalance,liquidity,microprice,mid_price_change,trade_count,buy_count,sell_count,total_trade_volume,buy_volume,sell_volume,trade_imbalance,trade_flow_ratio,avg_trade_size,max_trade_size,min_trade_size,std_trade_size,vwap
0,1773103232775,69154.000000,69173.400000,2.385600,1.387990,69163.700000,19.400000,0.264366,3.773590,69161.135647,-0.750000,2,0,2,0.085560,0.000000,0.085560,-1.000000,0.000000,0.042780,0.083550,0.002010,0.057657,69154.010000
